# Phase 2 reveal: GRCODE 23663 (National American Ins Co, WC)
This is the only notebook that reads the hidden answer key (`data/hidden/wc_23663_hidden.csv`). Everything before it (`03_one_company.ipynb`, `excel/wc_23663.xlsx`, `notes/phase2_prediction.md`) used known cells only and is frozen at tag `phase2-freeze`.


# Step 10. Reveal and grade
Question: How close was each method to what actually happened by 120 months, and how did my written prediction hold up?

The estimates, the Python reconciliation and the prediction (`notes/phase2_prediction.md`) are frozen in commit `e3c4780`, tag `phase2-freeze`. This step opens the hidden answer key for the first time.

Grading rules (set in the project plan before the reveal):
- **Matching basis:** paid methods are graded against actual paid at 120; reported methods against actual reported at 120.
- **Unpaid** on both bases = losses at 120 − paid to date at 12/31/2007. Reported unpaid includes case still open at 120.
- **Error on the reserve:** error = estimated unpaid − actual unpaid. Two scales: % of reserve = error ÷ actual unpaid; % of EP = error ÷ earned premium. Negative = understated.
- **AY 1998 is not graded.** It was already at 120 months on 12/31/2007, so its actual unpaid was known, not predicted. Graded totals cover AYs 1999–2007. The Excel totals (row 18) include AY 1998, so they differ from the graded estimated totals by AY 1998's unpaid (largest for EC industry, where the industry ELR isn't AY 1998's own ratio).
- **Selections:** read from the workbook (Part 4, row 40), the same frozen picks reconciled in step 8.

Sections: 10.1 open and verify the key · 10.2 actuals · 10.3 grade the 5 methods · 10.4 judgment vs the 6 averages · 10.5 score the 6 prediction calls.

## 10.1 Open and verify the answer key
Three checks before any value is used:
1. **Hash:** the file's SHA-256 must equal the hash printed in step 3. A match proves the file is byte-for-byte what was written before the Excel work started.
2. **Shape:** 45 rows, every one after 12/31/2007 (AY + lag − 1 > 2007).
3. **Source:** the key must equal the same company's hidden rows in `wc_clean.csv`, so grading from the key or from the clean data gives the same answer.

Prints the hash, the row count and the result of each check. No loss values are printed here.

In [1]:
# Phase 2, step 10.1: open the answer key and verify it
import sys; sys.path.insert(0, "..")
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from src import methods as m

GRCODE = 23663
KEY = Path("../data/hidden/wc_23663_hidden.csv")
STEP3_SHA = "cc9c2b0a48ef282d0a932c3ba7d83536988eaca0ba8e840e2b33dfe9be9881aa"
GRADED = list(range(1999, 2008))   # AY 1998 not graded (known at 12/31/2007)
METHODS = ["cl", "ec_company", "ec_industry", "bf_company", "bf_industry"]
LABELS = {"cl": "Chain ladder", "ec_company": "EC company", "ec_industry": "EC industry",
          "bf_company": "BF company", "bf_industry": "BF industry"}

# 1. Hash
sha = hashlib.sha256(KEY.read_bytes()).hexdigest()
print(f"SHA-256 now:    {sha}\nSHA-256 step 3: {STEP3_SHA}")
assert sha == STEP3_SHA, "Answer key changed since step 3; stop"

key = pd.read_csv(KEY)

# 2. Shape
assert len(key) == 45 and (key.GRCODE == GRCODE).all()
assert (key.AccidentYear + key.DevelopmentLag - 1 > m.VALUATION_YEAR).all()

# 3. Source: key == hidden rows of wc_clean.csv for this company
wc = pd.read_csv("../data/clean/wc_clean.csv")
line = wc[wc.GRCODE == GRCODE]
src = line[line.AccidentYear + line.DevelopmentLag - 1 > m.VALUATION_YEAR]
src = src.assign(Reported=src.IncurredLosses - src.BulkLoss)
cols = ["AccidentYear", "DevelopmentLag", "CumPaidLoss", "Reported"]
same = (key[cols].sort_values(cols[:2]).reset_index(drop=True)
        .equals(src[cols].sort_values(cols[:2]).reset_index(drop=True)))
assert same, "Answer key differs from wc_clean.csv hidden rows; stop"

print(f"Hash match: True | rows: {len(key)}, all after 2007: True | matches wc_clean.csv: True")

SHA-256 now:    cc9c2b0a48ef282d0a932c3ba7d83536988eaca0ba8e840e2b33dfe9be9881aa
SHA-256 step 3: cc9c2b0a48ef282d0a932c3ba7d83536988eaca0ba8e840e2b33dfe9be9881aa
Hash match: True | rows: 45, all after 2007: True | matches wc_clean.csv: True


## 10.2 Actuals at 120 months
For each AY: actual paid and reported at 120, actual unpaid on each basis (actual at 120 − paid to date), and the actual 120-month loss ratios (÷ EP). AYs 1999–2007 come from the key; AY 1998 from the known triangle.

Columns: `paid_to_date` = paid at 12/31/2007 · `paid_120`, `reported_120` = actual losses at 120 · `unpaid_paid`, `unpaid_reported` = actual unpaid per basis · `lr_paid`, `lr_reported` = actual loss ratio at 120. All $000 except the loss ratios. Totals row = graded AYs 1999–2007.

In [2]:
# Phase 2, step 10.2: actual losses at 120 and actual unpaid, both bases
k10 = key[key.DevelopmentLag == m.TARGET_LAG].set_index("AccidentYear")
ep = m.premium(line)
paid_to_date = m.latest(m.triangle(line, "paid"))["value"]

actual = pd.DataFrame({"ep": ep, "paid_to_date": paid_to_date})
actual["paid_120"] = k10.CumPaidLoss.reindex(actual.index)
actual["reported_120"] = k10.Reported.reindex(actual.index)
actual.loc[m.FIRST_AY, "paid_120"] = m.triangle(line, "paid").at[m.FIRST_AY, m.TARGET_LAG]
actual.loc[m.FIRST_AY, "reported_120"] = m.triangle(line, "reported").at[m.FIRST_AY, m.TARGET_LAG]
assert actual.notna().all().all()
actual["unpaid_paid"] = actual.paid_120 - actual.paid_to_date
actual["unpaid_reported"] = actual.reported_120 - actual.paid_to_date
actual["lr_paid"] = actual.paid_120 / actual.ep
actual["lr_reported"] = actual.reported_120 / actual.ep

show = actual.copy()
tot = actual.loc[GRADED].sum()
tot[["lr_paid", "lr_reported"]] = [tot.paid_120 / tot.ep, tot.reported_120 / tot.ep]
show.loc["1999-2007"] = tot
fmt = {c: "{:,.0f}" for c in show.columns}
fmt.update({"lr_paid": "{:.1%}", "lr_reported": "{:.1%}"})
print("ACTUALS AT 120 MONTHS ($000; loss ratios = losses at 120 / EP)")
print(show.apply(lambda s: s.map(fmt[s.name].format)).to_string())

ACTUALS AT 120 MONTHS ($000; loss ratios = losses at 120 / EP)
                   ep paid_to_date paid_120 reported_120 unpaid_paid unpaid_reported lr_paid lr_reported
AccidentYear                                                                                            
1998            9,937        8,353    8,353        8,622           0             269   84.1%       86.8%
1999           29,244       30,786   31,159       31,300         373             514  106.5%      107.0%
2000           21,159       16,218   16,230       16,236          12              18   76.7%       76.7%
2001           13,139       10,753   10,905       10,931         152             178   83.0%       83.2%
2002           14,809        6,777    7,051        7,079         274             302   47.6%       47.8%
2003           16,378        8,914    9,416        9,447         502             533   57.5%       57.7%
2004           17,372        5,814    6,655        6,748         841             934   38.3%     

## 10.3 Grade the 5 methods
For each basis, three tables:
1. **Total unpaid, AYs 1999–2007:** estimate, actual, error ($000), % of reserve, % of EP, and rank by absolute error (1 = closest). This is the table calls 1–3 are scored on.
2. **% of reserve error by AY** (error ÷ actual unpaid).
3. **% of EP error by AY** (error ÷ EP). Use this one where actual unpaid is small, because % of reserve explodes as the denominator nears zero.

Negative = the method understated the reserve.

In [3]:
# Phase 2, step 10.3: grade the 5 methods on each basis (matching basis, AYs 1999-2007)
xl = load_workbook("../excel/wc_23663.xlsx", data_only=True)
TRI_SHEET = {"paid": "Paid Triangle", "reported": "Reported Triangle"}

def selections(basis):
    """Frozen judgment selections, Part 4 row 40 (B40:J40)."""
    ws = xl[TRI_SHEET[basis]]
    return {k: float(ws.cell(row=40, column=k + 1).value) for k in range(1, m.TARGET_LAG)}

def total_grade(est_unpaid, act_unpaid, ep_):
    """Grade one total: estimate, actual, error, % of reserve, % of EP."""
    e, a, p = est_unpaid.sum(), act_unpaid.sum(), ep_.sum()
    return {"estimate": e, "actual": a, "error": e - a, "pct_reserve": (e - a) / a, "pct_ep": (e - a) / p}

def pct(df):
    return df.apply(lambda s: s.map("{:+.1%}".format))

results, by_ay = {}, {}
for basis in ["paid", "reported"]:
    elrs = {"company": m.elr_company(line, basis), "industry": m.elr_industry(wc, basis)}
    proj = m.project(line, basis, selections(basis), elrs).loc[GRADED]
    act = actual.loc[GRADED, f"unpaid_{basis}"]
    g = {mt: m.grade(proj[f"unpaid_{mt}"], act, proj.ep) for mt in METHODS}
    by_ay[basis] = g
    t = pd.DataFrame({LABELS[mt]: total_grade(proj[f"unpaid_{mt}"], act, proj.ep) for mt in METHODS}).T
    t["rank"] = t.error.abs().rank().astype(int)
    results[basis] = t

    print(f"\n{basis.upper()} BASIS - total unpaid, AYs 1999-2007 ($000; rank 1 = smallest |error|)")
    out = t.copy()
    for c in ["estimate", "actual", "error"]:
        out[c] = out[c].map("{:,.0f}".format)
    for c in ["pct_reserve", "pct_ep"]:
        out[c] = out[c].map("{:+.1%}".format)
    print(out.sort_values("rank").to_string())

    print(f"\n{basis.upper()} BASIS - % of reserve error by AY (error / actual unpaid)")
    print(pct(pd.DataFrame({LABELS[mt]: g[mt].pct_reserve for mt in METHODS})).to_string())
    print(f"\n{basis.upper()} BASIS - % of EP error by AY (error / EP)")
    print(pct(pd.DataFrame({LABELS[mt]: g[mt].pct_ep for mt in METHODS})).to_string())


PAID BASIS - total unpaid, AYs 1999-2007 ($000; rank 1 = smallest |error|)
             estimate  actual   error pct_reserve  pct_ep  rank
Chain ladder   13,058  13,659    -601       -4.4%   -0.4%     1
EC industry    12,383  13,659  -1,276       -9.3%   -0.8%     2
BF industry    17,545  13,659   3,886      +28.5%   +2.4%     3
BF company     22,978  13,659   9,319      +68.2%   +5.8%     4
EC company     44,539  13,659  30,880     +226.1%  +19.1%     5

PAID BASIS - % of reserve error by AY (error / actual unpaid)
             Chain ladder EC company EC industry BF company BF industry
AccidentYear                                                           
1999                -9.2%   -1763.2%    -3321.3%     -28.3%      -45.2%
2000             +3231.2%  +12968.0%   -22075.0%   +3465.5%    +2622.5%
2001              +179.5%     +91.8%    -1626.1%    +176.1%     +110.9%
2002               +38.8%   +1969.8%     +895.7%    +141.5%      +84.4%
2003               +59.7%    +866.8%     +218

## 10.4 Judgment vs the 6 averages (chain ladder)
Reruns chain ladder with each of the 6 averages in place of my selections and grades the total unpaid (AYs 1999–2007) the same way. Shows whether judgment helped or hurt against each mechanical rule. Call 6 compares against volume-weighted all years only (the benchmark fixed in the prediction); the other five are context only: picking the best of them after seeing the results would be hindsight, so they don't score anything.

Columns: estimate and error in $000, % of reserve, % of EP, rank by |error| among the 7 rows (1 = closest).

In [4]:
# Phase 2, step 10.4: chain ladder with judgment vs each of the 6 averages, both bases
cl_compare = {}
for basis in ["paid", "reported"]:
    tri = m.triangle(line, basis)
    act = actual.loc[GRADED, f"unpaid_{basis}"]
    picks = {"Judgment (frozen)": selections(basis)}
    for kind, n in m.AVERAGES:
        name = f"{'Simple' if kind == 'simple' else 'Volume-wtd'} {'all years' if n is None else f'latest {n}'}"
        picks[name] = m.average(tri, kind, n).to_dict()
    rows = {}
    for name, sel in picks.items():
        proj = m.project(line, basis, sel, {}).loc[GRADED]
        rows[name] = total_grade(proj.unpaid_cl, act, proj.ep)
    t = pd.DataFrame(rows).T
    t["rank"] = t.error.abs().rank().astype(int)
    cl_compare[basis] = t

    out = t.copy()
    for c in ["estimate", "actual", "error"]:
        out[c] = out[c].map("{:,.0f}".format)
    for c in ["pct_reserve", "pct_ep"]:
        out[c] = out[c].map("{:+.1%}".format)
    print(f"\n{basis.upper()} BASIS - chain ladder total unpaid, AYs 1999-2007 ($000; rank 1 = smallest |error|)")
    print(out.to_string())


PAID BASIS - chain ladder total unpaid, AYs 1999-2007 ($000; rank 1 = smallest |error|)
                     estimate  actual  error pct_reserve pct_ep  rank
Judgment (frozen)      13,058  13,659   -601       -4.4%  -0.4%     5
Simple all years       14,971  13,659  1,312       +9.6%  +0.8%     7
Simple latest 5        13,721  13,659     62       +0.5%  +0.0%     2
Simple latest 3        13,606  13,659    -53       -0.4%  -0.0%     1
Volume-wtd all years   14,006  13,659    347       +2.5%  +0.2%     4
Volume-wtd latest 5    12,782  13,659   -877       -6.4%  -0.5%     6
Volume-wtd latest 3    13,356  13,659   -303       -2.2%  -0.2%     3

REPORTED BASIS - chain ladder total unpaid, AYs 1999-2007 ($000; rank 1 = smallest |error|)
                     estimate  actual  error pct_reserve pct_ep  rank
Judgment (frozen)      16,091  14,436  1,655      +11.5%  +1.0%     1
Simple all years       23,068  14,436  8,632      +59.8%  +5.3%     7
Simple latest 5        20,983  14,436  6,547    

## 10.5 Score the prediction
Each call is scored exactly as its "Graded by" column says in `notes/phase2_prediction.md`. `Result` is Hit or Miss; `Evidence` gives the numbers behind it. Call 6 is scored separately on each basis.

In [5]:
# Phase 2, step 10.5: score the 6 calls in notes/phase2_prediction.md
paid_t, rep_t = results["paid"], results["reported"]
calls = []

# 1-2: closest total unpaid among the 5 methods is BF industry
for n, basis, t in [(1, "Paid", paid_t), (2, "Reported", rep_t)]:
    best = t.error.abs().idxmin()
    calls.append((n, f"{basis}: closest total unpaid", "BF industry", best == "BF industry",
                  f"closest = {best} ({t.loc[best, 'pct_reserve']:+.1%}); "
                  f"BF industry {t.loc['BF industry', 'pct_reserve']:+.1%}, rank {t.loc['BF industry', 'rank']}"))

# 3: paid CL total understates by more than 10%
cl = paid_t.loc["Chain ladder", "pct_reserve"]
calls.append((3, "Paid CL total", "Understates by > 10%", cl < -0.10, f"paid CL error {cl:+.1%} of reserve"))

# 4: company paid ELR overstates every AY 2002-2007
elr_co = m.elr_company(line, "paid")
lr = actual.loc[2002:2007, "lr_paid"]
calls.append((4, f"Company ELR ({elr_co:.1%}), AYs 2002-2007", "Overstates", bool((lr < elr_co).all()),
              "actual paid LR at 120: " + ", ".join(f"{ay} {v:.1%}" for ay, v in lr.items())))

# 5: AY 2007, paid CL closer than reported CL (% of reserve, each on its own basis)
p07 = by_ay["paid"]["cl"].loc[2007, "pct_reserve"]
r07 = by_ay["reported"]["cl"].loc[2007, "pct_reserve"]
calls.append((5, "AY 2007, paid vs reported CL", "Paid closer", abs(p07) < abs(r07),
              f"paid CL {p07:+.1%}, reported CL {r07:+.1%}"))

# 6: judgment loses to volume-weighted all years on CL total unpaid, each basis
for basis in ["paid", "reported"]:
    t = cl_compare[basis]
    j, v = t.loc["Judgment (frozen)", "error"], t.loc["Volume-wtd all years", "error"]
    calls.append((6, f"{basis.capitalize()}: judgment vs vol-wtd all years", "Lose", abs(v) < abs(j),
                  f"judgment {t.loc['Judgment (frozen)', 'pct_reserve']:+.1%}, "
                  f"vol-wtd all {t.loc['Volume-wtd all years', 'pct_reserve']:+.1%} of reserve"))

score = pd.DataFrame(calls, columns=["#", "Call", "Pick", "Hit", "Evidence"])
score["Result"] = score.Hit.map({True: "Hit", False: "Miss"})
print("PREDICTION SCORECARD")
print(score[["#", "Call", "Pick", "Result", "Evidence"]].to_string(index=False))
print(f"\n{score.Hit.sum()} of {len(score)} scored (call 6 counts once per basis)")

PREDICTION SCORECARD
 #                                    Call                 Pick Result                                                                                      Evidence
 1              Paid: closest total unpaid          BF industry   Miss                                    closest = Chain ladder (-4.4%); BF industry +28.5%, rank 3
 2          Reported: closest total unpaid          BF industry   Miss                                   closest = Chain ladder (+11.5%); BF industry +24.9%, rank 2
 3                           Paid CL total Understates by > 10%   Miss                                                                paid CL error -4.4% of reserve
 4      Company ELR (84.1%), AYs 2002-2007           Overstates    Hit actual paid LR at 120: 2002 47.6%, 2003 57.5%, 2004 38.3%, 2005 41.4%, 2006 46.5%, 2007 54.6%
 5            AY 2007, paid vs reported CL          Paid closer    Hit                                                            paid CL -18.1%, reported

Finding (write after reading 10.2–10.5): Chain ladder produced the closest estimates of the actual unpaid losses using both the paid and reported basis. Three of my seven predictions hit. Prediction 6 on the reported basis missed in my favor: my selections (+11.5%) beat volume-weighted all years (+53.2%). However, the misses in predictions 1-3 are signficant. I chose BF to produce a more accurate estimation of unpaid claims because I was betting on the immature AYs to have significant late emergence similar to AY 1998. 